# Notebook 6 - Real-Time Earthquake Monitoring

## Objectives

- Download the latest earthquakes
- Display them on an interactive map
- Color-code by magnitude
- Save the live dataset

In [1]:
import requests
import pandas as pd
import folium

In [2]:
url = (
    "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/"
    "all_day.geojson"
)

response = requests.get(url)

data = response.json()

In [3]:
records = []

for feature in data["features"]:

    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]

    records.append({
        "Location": props["place"],
        "Magnitude": props["mag"],
        "Time": props["time"],
        "Longitude": coords[0],
        "Latitude": coords[1],
        "Depth_km": coords[2]
    })

live_df = pd.DataFrame(records)

live_df.head()

,Location,Magnitude,Time,Longitude,Latitude,Depth_km
0,"10 km WNW of Cobb, CA",0.26,1784817562790,-122.831833,38.855000,1.63
1,"10 km WNW of Cobb, CA",0.76,1784817512530,-122.836830,38.854668,0.13
2,"10 km WNW of Cobb, CA",0.46,1784817304190,-122.836502,38.853668,1.61
3,"11 km WNW of Cobb, CA",1.84,1784817186480,-122.839996,38.853832,1.02
4,"7 km W of Tecate, B.C., MX",0.96,1784816707660,-116.707833,32.575000,3.01


In [4]:
live_df["Time"] = pd.to_datetime(
    live_df["Time"],
    unit="ms"
)

live_df.head()

,Location,Magnitude,Time,Longitude,Latitude,Depth_km
0,"10 km WNW of Cobb, CA",0.26,2026-07-23 14:39:22.790,-122.831833,38.855000,1.63
1,"10 km WNW of Cobb, CA",0.76,2026-07-23 14:38:32.530,-122.836830,38.854668,0.13
2,"10 km WNW of Cobb, CA",0.46,2026-07-23 14:35:04.190,-122.836502,38.853668,1.61
3,"11 km WNW of Cobb, CA",1.84,2026-07-23 14:33:06.480,-122.839996,38.853832,1.02
4,"7 km W of Tecate, B.C., MX",0.96,2026-07-23 14:25:07.660,-116.707833,32.575000,3.01


In [5]:
earthquake_map = folium.Map(
    location=[0, 0],
    zoom_start=2
)

In [6]:
for _, row in live_df.iterrows():

    magnitude = row["Magnitude"]

    if pd.isna(magnitude):
        magnitude = 0

    if magnitude < 3:
        color = "green"
    elif magnitude < 5:
        color = "orange"
    else:
        color = "red"

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],
        radius=max(3, magnitude * 2),
        color=color,
        fill=True,
        fill_color=color,
        popup=(
            f"{row['Location']}<br>"
            f"Magnitude: {magnitude}<br>"
            f"Depth: {row['Depth_km']} km"
        )
    ).add_to(earthquake_map)

In [7]:
earthquake_map

In [8]:
earthquake_map.save(
    "outputs/maps/live_earthquake_map.html"
)

print("Live map saved successfully.")

Live map saved successfully.


In [9]:
live_df.to_csv(
    "data/processed/live_earthquakes.csv",
    index=False
)

print("Live earthquake dataset saved.")

Live earthquake dataset saved.
